# Image Classifier — a convolutional neural network in Keras

Trains a CNN to sort images into the two classes given by the folders they were
loaded from.

This notebook is from 2022 and is preserved as it was written. Markdown and
comments were added later so the workflow can be followed end to end. The full
write-up, with diagrams, is in the [README](../README.md).

## What happens here

| Stage | Cells | What it produces |
|---|---|---|
| Set up the data generator | 3–6 | 3,309 images, resized and augmented on the fly |
| Build the network | 7–23 | A 3.45M-parameter CNN |
| Compile | 24 | Adam optimiser, binary cross-entropy loss |
| Train | 28 | 25 epochs over the image set |
| Predict and save | 29–43 | Predictions on the training data, model pickled |

## The task

Binary image classification. The model reads a 150×150 RGB image and outputs a
single number between 0 and 1 — the probability that it belongs to class 1.
The two classes are simply the two subfolders the images were loaded from.

## Running it

The image dataset is not included in this repository — see
[`data/README.md`](../data/README.md). The saved outputs below are from the
original run, so the workflow can be read through without it.

---

## 1 · Imports

<!-- annotated -->

In [1]:
import numpy as np

In [2]:
import pandas as pd

In [3]:
import os

## 2 · Point at the image folders

`ImageDataGenerator.flow_from_directory` infers the class labels from the
subfolder names, so a directory laid out like this needs no label file:

```
data/
├── class_a/    ← every image in here gets label 0
└── class_b/    ← every image in here gets label 1
```

The path below is the local machine it was written on; substitute your own.

<!-- annotated -->

In [4]:
# Local path to the image folders; one subfolder per class.
dir=r'C:\Users\abalu\Desktop\data'

## 3 · Configure the data generator

`ImageDataGenerator` streams images from disk in batches rather than loading
all 3,309 into memory at once, and applies random transformations as it goes.

Each epoch therefore sees slightly different versions of the same photographs —
flipped, shifted, zoomed — which multiplies the effective size of the dataset
and makes the network less able to memorise individual images.

| Setting | Value | Unit | Effect |
|---|---|---|---|
| `rescale` | 1/255 | factor | maps pixel values from 0–255 to 0–1 |
| `horizontal_flip` | True | on/off | mirrors the image left to right |
| `width_shift_range` | 0.2 | fraction | slides up to 20% horizontally |
| `height_shift_range` | 0.2 | fraction | slides up to 20% vertically |
| `shear_range` | 0.2 | degrees | slants the image |
| `rotation_range` | 0.2 | degrees | rotates the image |
| `zoom_range` | 0.2 | fraction | scales between 0.8× and 1.2× |

Note the units are not uniform: the shift and zoom ranges are *fractions* of
the image, while shear and rotation are measured in *degrees*.

These transforms are reproduced in
[`src/augmentation.py`](../src/augmentation.py) and illustrated in the
[README](../README.md).

<!-- annotated -->

In [5]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [6]:
# Augmentation: random flips, shifts, shears and zooms on every epoch.
train_gen=ImageDataGenerator(
    rescale=1./255,
    horizontal_flip=True,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range= 0.2,
    rotation_range=0.2,
    zoom_range=0.2
)

### Load the images

`target_size=(150, 150)` resizes every image to a fixed square, because a
network's dense layers need a fixed input size. `class_mode='binary'` produces
a single 0/1 label per image rather than a one-hot vector.

The output records what was found: **3,309 images across 2 classes**.

<!-- annotated -->

In [7]:
# Resize everything to 150x150; labels come from the folder names.
train_data=train_gen.flow_from_directory(
dir,
target_size=(150,150),
batch_size=32,
class_mode='binary'
)

Found 3309 images belonging to 2 classes.


## 4 · Build the network

The architecture is four convolution/pooling blocks followed by a classifier
head. Each block detects patterns at a coarser scale than the last:

| Block | Filters | Output size |
|---|---|---|
| 1 | 32 | 74×74 |
| 2 | 64 | 36×36 |
| 3 | 128 | 17×17 |
| 4 | 128 | 7×7 |

**What a convolution does.** It slides a small 3×3 window across the image,
computing a weighted sum at each position. The weights are learned, so early
layers converge on simple detectors — edges, corners, colour transitions —
while later layers combine those into larger structures.

**What pooling does.** `MaxPooling2D((2,2))` takes the strongest value in each
2×2 square, halving the width and height. This discards precise position while
keeping the presence of a feature, which is what makes the network tolerant of
small shifts.

The pattern of shrinking spatially while growing in depth is the standard
convolutional trade: less *where*, more *what*.

This architecture is reconstructed analytically in
[`src/architecture.py`](../src/architecture.py), whose tests assert it matches
the `model.summary()` output below exactly.

<!-- annotated -->

In [8]:
import tensorflow as tf

In [9]:
import tensorflow.keras as kt

In [10]:
model=kt.models.Sequential()

In [11]:
# 32 filters, 3x3 window -> output 148x148 (valid padding loses 2px).
model.add(kt.layers.Conv2D(32,(3,3),input_shape=(150,150,3),activation='relu'))

In [12]:
# Halve both dimensions: 148x148 -> 74x74.
model.add(kt.layers.MaxPooling2D((2,2)))

In [13]:
# Deeper: 64 filters over the previous 32 channels.
model.add(kt.layers.Conv2D(64,(3,3),activation='relu'))

In [14]:
model.add(kt.layers.MaxPooling2D((2,2)))

In [15]:
# Deeper still: 128 filters.
model.add(kt.layers.Conv2D(128,(3,3),activation='relu'))

In [16]:
model.add(kt.layers.MaxPooling2D((2,2)))

In [17]:
model.add(kt.layers.Conv2D(128,(3,3),activation='relu'))

In [18]:
model.add(kt.layers.MaxPooling2D((2,2)))

### Flatten and classify

`Flatten` unrolls the final 7×7×128 feature map into a single vector of
**6,272** values. `Dropout(0.5)` then randomly zeroes half of them on each
training pass, forcing the network to spread its reasoning across many features
rather than depending on a few.

The `Dense(512)` layer that follows connects all 6,272 inputs to 512 units,
which costs 3,211,776 parameters — 93% of the whole model. The final
`Dense(1, activation='sigmoid')` squeezes those 512 values into one probability.

<!-- annotated -->

In [19]:
# Unroll the final 7x7x128 feature map into 6,272 values.
model.add(kt.layers.Flatten())

In [20]:
# Zero half the activations during training to reduce overfitting.
model.add(kt.layers.Dropout(0.5))

In [21]:
# The largest layer: 6,272 x 512 = 3.2M parameters.
model.add(kt.layers.Dense(512,activation='relu'))

In [22]:
# One sigmoid output = probability of class 1.
model.add(kt.layers.Dense(1,activation='sigmoid'))

### The parameter count

`model.summary()` reports **3,453,121 trainable parameters** across the whole
network, and shows how the tensor shrinks from 150×150 down to 7×7.

<!-- annotated -->

In [23]:
# 3,453,121 trainable parameters in total.
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 148, 148, 32)      896       
                                                                 
 max_pooling2d (MaxPooling2D  (None, 74, 74, 32)       0         
 )                                                               
                                                                 
 conv2d_1 (Conv2D)           (None, 72, 72, 64)        18496     
                                                                 
 max_pooling2d_1 (MaxPooling  (None, 36, 36, 64)       0         
 2D)                                                             
                                                                 
 conv2d_2 (Conv2D)           (None, 34, 34, 128)       73856     
                                                                 
 max_pooling2d_2 (MaxPooling  (None, 17, 17, 128)      0

## 5 · Compile

`binary_crossentropy` is the standard loss for a two-class problem with a
sigmoid output: it penalises confident wrong answers much more heavily than
uncertain ones. `adam` is an adaptive optimiser that adjusts the learning rate
per parameter as training proceeds.

<!-- annotated -->

In [24]:
# Binary cross-entropy is the standard loss for two-class problems.
model.compile(loss='binary_crossentropy',optimizer='adam',metrics=['accuracy'])

In [25]:
from sklearn.model_selection import train_test_split

In [26]:
train_data

## 6 · Train

25 epochs, 100 batches of 32 images each — so roughly 3,200 images per epoch,
approximately one full pass over the 3,309 available.

The output below is the original 2022 run. It is transcribed to
[`data/training_log_2022.csv`](../data/training_log_2022.csv) and charted in
the [README](../README.md).

At about 71 seconds per epoch, the run took roughly **30 minutes** on CPU.

<!-- annotated -->

In [27]:
# 25 epochs x 100 batches x 32 images ~= one pass over the data per epoch.
model.fit_generator(train_data,steps_per_epoch=100,epochs=25)

C:\Users\abalu\AppData\Local\Temp\ipykernel_21064\2770445033.py:1: UserWarning: `Model.fit_generator` is deprecated and will be removed in a future version. Please use `Model.fit`, which supports generators.
  model.fit_generator(train_data,steps_per_epoch=100,epochs=25)


Epoch 1/25
100/100 [==============================] - 70s 695ms/step - loss: 0.6746 - accuracy: 0.5753
Epoch 2/25
100/100 [==============================] - 70s 703ms/step - loss: 0.6577 - accuracy: 0.6165
Epoch 3/25
100/100 [==============================] - 71s 703ms/step - loss: 0.6454 - accuracy: 0.6334
Epoch 4/25
100/100 [==============================] - 72s 708ms/step - loss: 0.6317 - accuracy: 0.6495
Epoch 5/25
100/100 [==============================] - 71s 708ms/step - loss: 0.6347 - accuracy: 0.6504
Epoch 6/25
100/100 [==============================] - 71s 711ms/step - loss: 0.6337 - accuracy: 0.6419
Epoch 7/25
100/100 [==============================] - 71s 711ms/step - loss: 0.6323 - accuracy: 0.6394
Epoch 8/25
100/100 [==============================] - 71s 708ms/step - loss: 0.6338 - accuracy: 0.6454
Epoch 9/25
100/100 [==============================] - 70s 701ms/step - loss: 0.6305 - accuracy: 0.6558
Epoch 10/25
100/100 [==============================] - 72s 716ms/step - l

## 7 · Save the model

`pickle` writes the trained model to disk so it can be reloaded without
retraining. Keras also offers its own `model.save()`, which stores the
architecture, weights, and optimiser state in a portable format — generally the
better choice for Keras models.

<!-- annotated -->

In [28]:
import pickle

In [30]:
# Persist the trained model so it can be reloaded without retraining.
pickle.dump(model,open('modelman.pkl','wb'))

INFO:tensorflow:Assets written to: ram://5c5dac71-70b0-407b-ae7f-c14ab4505d0c/assets


INFO:tensorflow:Assets written to: ram://5c5dac71-70b0-407b-ae7f-c14ab4505d0c/assets


## 8 · Predict

Runs the trained model over the image set and converts its output probabilities
into class labels.

The model emits one float per image, such as `0.574` or `0.838`. Rounding at a
threshold of 0.5 turns each into a 0 or a 1.

<!-- annotated -->

In [31]:
# Model outputs one probability per image.
y=model.predict(train_data)

104/104 [==============================] - 56s 532ms/step


In [36]:
y=np.round(y)

In [39]:
y=np.argmax(y,axis=1)

In [40]:
y

array([0, 0, 0, ..., 0, 0, 0], dtype=int64)

In [44]:
y_pred=model.predict(train_data)

104/104 [==============================] - 54s 521ms/step


In [45]:
y_pred

array([[0.57438093],
       [0.8379696 ],
       [0.83180934],
       ...,
       [0.492071  ],
       [0.4697778 ],
       [0.84977025]], dtype=float32)

In [46]:
y_pred=np.round(y_pred,3)

In [47]:
y_pred

array([[0.574],
       [0.838],
       [0.832],
       ...,
       [0.492],
       [0.47 ],
       [0.85 ]], dtype=float32)

In [48]:
y_pred=np.round(y_pred,1)

In [49]:
y_pred

array([[0.6],
       [0.8],
       [0.8],
       ...,
       [0.5],
       [0.5],
       [0.8]], dtype=float32)

In [50]:
# Round at 0.5 to turn probabilities into class labels.
y_pred=np.round(y_pred)

In [52]:
y_pred.astype(int)

array([[1],
       [1],
       [1],
       ...,
       [0],
       [0],
       [1]])

In [53]:
pickle.dump(model,open('model.pkl','wb'))

INFO:tensorflow:Assets written to: ram://e1e1addb-ac64-414b-8146-40ec0e5b2a90/assets


INFO:tensorflow:Assets written to: ram://e1e1addb-ac64-414b-8146-40ec0e5b2a90/assets


---

## 9 · Results

| Metric | Epoch 1 | Epoch 25 |
|---|---|---|
| Training accuracy | 0.5753 | **0.7581** |
| Training loss | 0.6746 | **0.4923** |

Accuracy rose from roughly chance (0.5753, where guessing scores 0.50) to
**0.7581** over 25 epochs, and the loss fell steadily from 0.6746 to 0.4923.
The curves were still improving when training stopped, so the model had not yet
converged.

**These are training figures.** `flow_from_directory` was pointed at a single
folder with no `validation_split`, so every image the model was scored on was
also an image it trained on. Measuring generalisation would need a held-out set
of images the model never saw — ideally split so that no individual appears in
both halves.

## What is in the rest of this project

| Path | What it holds |
|---|---|
| [`README.md`](../README.md) | The full walkthrough, with diagrams |
| [`src/architecture.py`](../src/architecture.py) | The layer shapes and parameter counts, tested against the summary above |
| [`src/augmentation.py`](../src/augmentation.py) | The augmentation transforms, reproduced and tested |
| [`data/training_log_2022.csv`](../data/training_log_2022.csv) | The recovered 25-epoch history |

## Scope

This model sorts images into the two folders it was trained on. Those folder
labels describe how the dataset was organised — they are not a judgement about
any person, and appearance does not determine identity.

A model trained on a limited image collection can just as easily learn lighting,
camera angle, hairstyle, background, or image source as anything else. Nothing
here should be used for identification, screening, access control, moderation,
or any decision about a person.


<!-- annotated -->